# 03. Preprocesamiento de datos

## Objetivo

En este notebook se prepara el conjunto de datos de Airbnb de Buenos Aires para el modelado. Se revisa la estructura, se decide cómo tratar los valores faltantes, se transforman variables categóricas y se crean características a partir de los baños y las amenities. Al final se exploran valores extremos de las variables numéricas.

**Variable objetivo del proyecto:** `price_quote_price_per_night`. Esta columna representa el precio por noche que se quiere predecir. En este notebook se conserva como target; en el notebook de entrenamiento se separará de las variables predictoras.

> Nota metodológica: las decisiones de limpieza deben basarse en el significado de los datos, no solamente en que un valor sea marcado como outlier por una regla estadística.


## 1. Cargar los datos

Se carga el dataset procesado que salió de la etapa de análisis exploratorio (`02_eda`). Se usa `df_dp` para distinguir este DataFrame de los utilizados en otros notebooks. La visualización inicial permite comprobar que el archivo se leyó correctamente.


In [297]:
import pandas as pd

df_dp = pd.read_csv("../data/processed/airbnb_eda.csv")
df_dp

,host_is_superhost,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,price_quote_price_per_night,minimum_nights,maximum_nights,availability_365,number_of_reviews,review_scores_rating
0,t,Palermo,-34.581993,-58.424028,Entire condo,Entire home/apt,2,1 bath,1.0,1.0,"[""Stove"", ""Private entrance"", ""Extra pillows a...",186530.00,3.0,1125.0,119,52,4.83
1,f,Belgrano,-34.561650,-58.444080,Entire condo,Entire home/apt,2,1 bath,1.0,1.0,"[""Air conditioning"", ""Heating"", ""TV"", ""Doorman...",91729.00,2.0,30.0,209,1,5.00
2,t,Palermo,-34.581848,-58.441605,Entire rental unit,Entire home/apt,4,1.5 baths,2.0,2.0,"[""Stove"", ""Extra pillows and blankets"", ""Iron""...",232074.07,2.0,1125.0,255,394,4.94
3,t,San Nicolas,-34.602540,-58.378110,Entire rental unit,Entire home/apt,3,1 bath,1.0,2.0,"[""Private entrance"", ""Extra pillows and blanke...",117408.00,1.0,30.0,161,393,4.67
4,f,Chacarita,-34.584940,-58.444190,Entire loft,Entire home/apt,2,1 bath,1.0,1.0,"[""Iron"", ""Essentials"", ""Carbon monoxide alarm""...",465014.50,12.0,1125.0,269,37,4.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29471,f,Monserrat,-34.610628,-58.387609,Entire rental unit,Entire home/apt,2,1 bath,1.0,1.0,"[""Air conditioning"", ""TV"", ""Exterior security ...",78989.00,3.0,365.0,360,0,NaN
29472,f,Retiro,-34.595273,-58.378580,Entire rental unit,Entire home/apt,2,1 bath,1.0,1.0,"[""Air conditioning"", ""TV"", ""Exterior security ...",88672.00,1.0,365.0,171,0,NaN
29473,f,Belgrano,-34.558800,-58.455169,Private room in casa particular,Private room,3,Private half-bath,1.0,2.0,"[""Hot water"", ""Lock on bedroom door"", ""Dedicat...",55037.33,1.0,365.0,87,0,NaN
29474,f,Belgrano,-34.562883,-58.454144,Entire rental unit,Entire home/apt,2,1 bath,1.0,1.0,"[""Kitchen"", ""Hot water"", ""Air conditioning"", ""...",107018.00,1.0,365.0,363,0,NaN


## 2. Inspeccionar la estructura y los tipos de datos

`info()` muestra la cantidad de registros, las columnas, sus tipos de datos y cuántos valores no nulos hay en cada variable. Esta revisión ayuda a detectar columnas con faltantes o tipos que podrían necesitar transformación antes del modelado.


In [298]:
df_dp.info()

<class 'pandas.DataFrame'>
RangeIndex: 29476 entries, 0 to 29475
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   host_is_superhost            29476 non-null  str    
 1   neighbourhood_cleansed       29476 non-null  str    
 2   latitude                     29476 non-null  float64
 3   longitude                    29476 non-null  float64
 4   property_type                29476 non-null  str    
 5   room_type                    29476 non-null  str    
 6   accommodates                 29476 non-null  int64  
 7   bathrooms_text               29454 non-null  str    
 8   bedrooms                     29463 non-null  float64
 9   beds                         28960 non-null  float64
 10  amenities                    29476 non-null  str    
 11  price_quote_price_per_night  27356 non-null  float64
 12  minimum_nights               29475 non-null  float64
 13  maximum_nights             

## 3. Tratamiento de valores faltantes

En esta versión se decidió eliminar las filas que tienen al menos un valor nulo mediante `dropna()`. Es una decisión simple y reproducible, pero reduce el tamaño del conjunto de datos y puede afectar su representatividad si los faltantes no se distribuyen al azar. Conviene comparar el `info()` anterior y posterior para cuantificar esa pérdida.

Esta decisión se mantiene por ahora, tal como se definió para el proyecto; no implica que eliminar filas sea siempre la mejor estrategia.


In [299]:
# eliminar todas las filas con valores nulos
df_dp = df_dp.dropna()
df_dp.info()

<class 'pandas.DataFrame'>
Index: 23926 entries, 0 to 29435
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   host_is_superhost            23926 non-null  str    
 1   neighbourhood_cleansed       23926 non-null  str    
 2   latitude                     23926 non-null  float64
 3   longitude                    23926 non-null  float64
 4   property_type                23926 non-null  str    
 5   room_type                    23926 non-null  str    
 6   accommodates                 23926 non-null  int64  
 7   bathrooms_text               23926 non-null  str    
 8   bedrooms                     23926 non-null  float64
 9   beds                         23926 non-null  float64
 10  amenities                    23926 non-null  str    
 11  price_quote_price_per_night  23926 non-null  float64
 12  minimum_nights               23926 non-null  float64
 13  maximum_nights               239

## 4. Revisar variables categóricas y conteos

Las siguientes celdas inspeccionan las categorías más frecuentes antes de transformarlas. Esto sirve para conocer los valores presentes y detectar etiquetas inesperadas.


### 4.1. Estado de superhost

Se cuentan los valores de `host_is_superhost` para confirmar las etiquetas originales (`t` y `f`) antes de convertirlas a formato numérico.

In [300]:
df_dp["host_is_superhost"].value_counts()


host_is_superhost
f    12833
t    11093
Name: count, dtype: int64

### 4.2. Tipo de propiedad

Se revisa la frecuencia de cada valor de `property_type`. Como es una variable nominal, no debe codificarse asignando números arbitrarios a cada categoría.

In [301]:
df_dp["property_type"].value_counts()


property_type
Entire rental unit                    17576
Entire condo                           2455
Entire serviced apartment               771
Private room in rental unit             658
Entire loft                             552
Private room in home                    488
Entire home                             324
Room in hotel                           188
Private room in condo                   140
Entire vacation home                    130
Private room in guesthouse               70
Private room in casa particular          65
Private room in bed and breakfast        62
Private room in serviced apartment       49
Entire townhouse                         40
Room in boutique hotel                   34
Private room in townhouse                32
Private room in villa                    29
Shared room in guesthouse                26
Shared room in home                      25
Entire guesthouse                        21
Room in aparthotel                       20
Shared room in hos

### 4.3. Tipo de habitación

Se cuentan las categorías de `room_type` para conocer la distribución de los tipos de alojamiento.

In [302]:
df_dp["room_type"].value_counts()


room_type
Entire home/apt    21941
Private room        1857
Shared room          101
Hotel room            27
Name: count, dtype: int64

### 4.4. Descripción original de baños

Se inspeccionan los textos de `bathrooms_text`. Esta columna mezcla cantidad y descripción (por ejemplo, si el baño es privado o compartido), por lo que luego se separará en características más útiles para el modelo.

In [303]:
df_dp["bathrooms_text"].value_counts()


bathrooms_text
1 bath               17571
1.5 baths             2310
2 baths               1537
1 private bath         588
1 shared bath          559
2.5 baths              357
3 baths                199
1.5 shared baths       159
2 shared baths         141
3 shared baths         112
0 baths                 64
3.5 baths               56
4 baths                 55
6 shared baths          29
0 shared baths          19
4 shared baths          19
Private half-bath       19
2.5 shared baths        16
Shared half-bath        15
5 baths                 14
4.5 baths               14
3.5 shared baths        13
6 baths                  9
Half-bath                9
9 baths                  6
5 shared baths           6
7 baths                  5
8 baths                  5
5.5 baths                3
8.5 baths                3
4.5 shared baths         2
5.5 shared baths         2
20 shared baths          2
8.5 shared baths         2
22 baths                 1
7 shared baths           1
9.5 baths    

### 4.5. Cantidad de dormitorios

Se revisa la frecuencia de `bedrooms`. Los valores poco frecuentes no son automáticamente errores; deben interpretarse según el contexto del alojamiento.

In [304]:
df_dp["bedrooms"].value_counts()


bedrooms
1.0     16143
2.0      3366
0.0      3280
3.0       821
4.0       188
5.0        50
7.0        27
6.0        18
8.0        16
9.0         6
11.0        4
14.0        3
22.0        1
10.0        1
31.0        1
29.0        1
Name: count, dtype: int64

### 4.6. Cantidad de camas

Se revisa la frecuencia de `beds`. Luego se puede contrastar con dormitorios y capacidad para investigar posibles inconsistencias.

In [305]:
df_dp["beds"].value_counts()

beds
1.0     12138
2.0      6241
3.0      3273
4.0      1353
5.0       455
6.0       221
7.0        77
8.0        50
9.0        31
10.0       21
12.0       17
11.0       11
16.0        9
14.0        6
15.0        5
13.0        4
20.0        3
17.0        2
25.0        2
22.0        2
28.0        1
30.0        1
50.0        1
18.0        1
27.0        1
Name: count, dtype: int64

### 4.7. Combinaciones de amenities

`value_counts()` en la columna `amenities` cuenta combinaciones completas de servicios, no la frecuencia de cada servicio individual. Para contar cada amenity por separado se usa `Counter` en la siguiente sección.

In [306]:
df_dp["amenities"].value_counts()

amenities
["Kitchen", "TV", "Wifi", "Air conditioning"]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       61
["Air conditioning", "TV", "Washer", "Kitchen", "Wifi"]                                

## 5. Contar amenities individuales

`amenities` está guardada como texto que representa una lista de servicios. `ast.literal_eval()` convierte cada representación en una lista de Python sin ejecutar código arbitrario. Luego `Counter` suma cuántas veces aparece cada amenity en los alojamientos.

Se utiliza este conteo para seleccionar las 20 amenities más frecuentes y crear una variable binaria para cada una. La selección por frecuencia reduce la cantidad de columnas, aunque las amenities menos frecuentes quedan fuera de esta representación.


In [307]:
from collections import Counter
import ast

amenities_list = (
    df_dp["amenities"]
    .dropna()
    .apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
)

counter = Counter(
    amenity.strip() for amenities in amenities_list for amenity in amenities
)

counter.most_common(20)

[('Kitchen', 22317),
 ('Wifi', 21447),
 ('Hot water', 20938),
 ('Dishes and silverware', 19517),
 ('Cooking basics', 19060),
 ('Hangers', 18715),
 ('Bed linens', 18586),
 ('Hair dryer', 17277),
 ('Refrigerator', 16868),
 ('Microwave', 16103),
 ('Bidet', 15866),
 ('Air conditioning', 15805),
 ('Essentials', 15605),
 ('Hot water kettle', 15545),
 ('Elevator', 15377),
 ('Room-darkening shades', 15192),
 ('TV', 14687),
 ('Dedicated workspace', 14407),
 ('Iron', 13988),
 ('Extra pillows and blankets', 13536)]

## 6. Transformar variables y crear características

En esta sección se realizan transformaciones deterministas:

- `host_is_superhost`: convierte `t`/`f` a `1`/`0`.
- Variables categóricas: `get_dummies()` crea una columna binaria por categoría para `property_type`, `room_type` y `neighbourhood_cleansed`.
- Baños: se extrae la cantidad numérica y se crean indicadores para baño privado, compartido o sin especificación. Los textos `Half-bath`, `Private half-bath` y `Shared half-bath` se representan como `0.5` baños para evitar que esos casos queden como `NaN`.
- Amenities: se crean indicadores `1`/`0` para las 20 amenities más frecuentes. La pertenencia se comprueba sobre listas reales para evitar coincidencias parciales de texto.

**Consideración para el modelado:** `get_dummies()` se usa aquí para preparar y explorar el dataset. En el pipeline de evaluación final es preferible ajustar `OneHotEncoder(handle_unknown="ignore")` solamente con los datos de entrenamiento, después del train/test split.


In [308]:
# Convertir host_is_superhost a 1 o 0
df_dp["host_is_superhost"] = df_dp["host_is_superhost"].map({"t": 1, "f": 0})

# Transformar las variables categóricas en columnas binarias
columnas_categoricas = ["property_type", "room_type", "neighbourhood_cleansed"]
df_dp = pd.get_dummies(df_dp, columns=columnas_categoricas, dtype=int)

# Extraer la cantidad numérica de baños
df_dp["bathrooms"] = (
    df_dp["bathrooms_text"].str.extract(r"(\d+\.?\d*)")[0].astype(float)
)

# Los textos half-bath no incluyen una cantidad numérica explícita; representarlos como 0.5
half_bath = df_dp["bathrooms_text"].str.contains("half-bath", case=False, na=False)
df_dp.loc[half_bath, "bathrooms"] = 0.5

# Crear columnas binarias según el tipo de baño
df_dp["bathroom_private"] = (
    df_dp["bathrooms_text"].str.contains("private", case=False, na=False).astype(int)
)
df_dp["bathroom_shared"] = (
    df_dp["bathrooms_text"].str.contains("shared", case=False, na=False).astype(int)
)
df_dp["bathroom_unspecified"] = (
    ~df_dp["bathrooms_text"].str.contains("private|shared", case=False, na=False)
).astype(int)

# Eliminar la columna original de descripción de baños
df_dp = df_dp.drop(columns=["bathrooms_text"])

# Convertir la representación textual de amenities en listas reales
df_dp["amenities"] = df_dp["amenities"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Seleccionar las 20 amenities más frecuentes y crear indicadores binarios
top_20_amenities = [amenity for amenity, count in counter.most_common(20)]
for amenity in top_20_amenities:
    df_dp[amenity] = df_dp["amenities"].apply(lambda lista: int(amenity in lista))

# Eliminar la columna original de amenities
df_dp = df_dp.drop(columns=["amenities"])


### 6.1. Revisar el DataFrame transformado

Se muestra una vista de los datos y se comprueba la estructura después de las transformaciones. Conviene verificar que las columnas originales reemplazadas ya no estén presentes y que las nuevas columnas binarias contengan valores `0` y `1`.


In [309]:
df_dp

,host_is_superhost,latitude,longitude,accommodates,bedrooms,beds,price_quote_price_per_night,minimum_nights,maximum_nights,availability_365,...,Bidet,Air conditioning,Essentials,Hot water kettle,Elevator,Room-darkening shades,TV,Dedicated workspace,Iron,Extra pillows and blankets
0,1,-34.581993,-58.424028,2,1.0,1.0,186530.00,3.0,1125.0,119,...,1,1,1,0,1,1,1,0,1,1
1,0,-34.561650,-58.444080,2,1.0,1.0,91729.00,2.0,30.0,209,...,0,1,0,0,1,0,1,0,0,0
2,1,-34.581848,-58.441605,4,2.0,2.0,232074.07,2.0,1125.0,255,...,1,0,1,0,1,1,0,1,1,1
3,1,-34.602540,-58.378110,3,1.0,2.0,117408.00,1.0,30.0,161,...,1,0,1,0,1,1,0,1,1,1
4,0,-34.584940,-58.444190,2,1.0,1.0,465014.50,12.0,1125.0,269,...,0,1,1,0,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29353,1,-34.573106,-58.441271,2,1.0,1.0,99373.00,1.0,365.0,355,...,1,1,1,1,1,0,1,1,0,0
29379,0,-34.602460,-58.388750,10,3.0,10.0,135300.10,1.0,365.0,355,...,0,1,1,1,1,1,1,1,1,1
29414,1,-34.585832,-58.406372,2,1.0,2.0,75982.50,2.0,365.0,339,...,1,1,0,1,1,1,1,1,0,1
29429,1,-34.570417,-58.432073,2,1.0,1.0,91729.00,1.0,1125.0,361,...,1,1,1,1,0,0,1,0,0,0


In [310]:
df_dp.info()

<class 'pandas.DataFrame'>
Index: 23926 entries, 0 to 29435
Columns: 137 entries, host_is_superhost to Extra pillows and blankets
dtypes: float64(9), int64(128)
memory usage: 25.2 MB


## 7. Detectar posibles outliers en variables numéricas

Se utiliza el criterio del rango intercuartílico (IQR). Para cada variable se calculan `Q1`, `Q3`, `IQR = Q3 - Q1` y los límites `Q1 - 1.5 × IQR` y `Q3 + 1.5 × IQR`. Los valores fuera de esos límites se cuentan como posibles outliers.

Esta tabla es diagnóstica: **no elimina registros**. El IQR puede ser poco informativo en variables discretas concentradas en un único valor (por ejemplo, si `Q1 = Q3`, el IQR es cero) o en distribuciones muy asimétricas. Por eso se interpretan los resultados con el significado de cada variable. El precio no se incluye aquí porque ya se trató en el análisis previo.


In [311]:
columnas_numericas = [
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
]

resultados = []

for columna in columnas_numericas:
    q1 = df_dp[columna].quantile(0.25)
    q3 = df_dp[columna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = (df_dp[columna] < limite_inferior) | (df_dp[columna] > limite_superior)

    resultados.append(
        {
            "variable": columna,
            "limite_inferior": limite_inferior,
            "limite_superior": limite_superior,
            "cantidad_outliers": outliers.sum(),
            "porcentaje_outliers": round(outliers.mean() * 100, 2),
        }
    )

df_outliers = pd.DataFrame(resultados).sort_values(
    "porcentaje_outliers", ascending=False
)

df_outliers

,variable,limite_inferior,limite_superior,cantidad_outliers,porcentaje_outliers
2,bedrooms,1.000,1.000,7783,32.53
1,bathrooms,1.000,1.000,5208,21.77
3,beds,-0.500,3.500,2274,9.50
4,minimum_nights,-2.000,6.000,2109,8.81
7,number_of_reviews,-78.500,149.500,1679,7.02
8,review_scores_rating,4.370,5.330,1462,6.11
0,accommodates,-1.000,7.000,357,1.49
5,maximum_nights,-937.500,2362.500,9,0.04
6,availability_365,-99.875,599.125,0,0.00


## 8. Inspeccionar distribuciones y valores frecuentes

Para las variables con más casos marcados, `describe()` resume la distribución e incluye percentiles altos (90 %, 95 %, 99 % y 99.5 %). `value_counts()` muestra los valores más frecuentes. Esta información permite distinguir valores extremos potencialmente legítimos de errores de captura o inconsistencias.


In [312]:
columnas_revisar = [
    "bedrooms",
    "bathrooms",
    "beds",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
]

for columna in columnas_revisar:
    print(f"\n--- {columna} ---")
    print(df_dp[columna].describe(percentiles=[0.90, 0.95, 0.99, 0.995]))
    print("\nValores más frecuentes:")
    print(df_dp[columna].value_counts().head(10))


--- bedrooms ---
count    23926.000000
mean         1.128354
std          0.860872
min          0.000000
90%          2.000000
95%          2.000000
99%          4.000000
99.5%        5.000000
max         31.000000
Name: bedrooms, dtype: float64

Valores más frecuentes:
bedrooms
1.0    16143
2.0     3366
0.0     3280
3.0      821
4.0      188
5.0       50
7.0       27
6.0       18
8.0       16
9.0        6
Name: count, dtype: int64

--- bathrooms ---
count    23926.000000
mean         1.208037
std          0.599939
min          0.000000
90%          2.000000
95%          2.000000
99%          3.500000
99.5%        4.000000
max         22.000000
Name: bathrooms, dtype: float64

Valores más frecuentes:
bathrooms
1.0    18718
1.5     2469
2.0     1678
2.5      373
3.0      311
0.0       83
4.0       74
3.5       69
0.5       43
6.0       38
Name: count, dtype: int64

--- beds ---
count    23926.000000
mean         1.918457
std          1.436295
min          1.000000
90%          3.000000

## 9. Investigar alojamientos con muchos dormitorios

Se inspeccionan las filas con más de ocho dormitorios junto con `accommodates` y `beds`. La finalidad es identificar registros que merecen revisión, no asumir que son errores automáticamente. Un alojamiento grande puede ser real; la coherencia con la capacidad y las camas aporta contexto adicional.


In [313]:
df_dp.loc[df_dp["bedrooms"] > 8, ["bedrooms", "accommodates", "beds"]].sort_values(
    "bedrooms", ascending=False
)

,bedrooms,accommodates,beds
18327,31.0,2,2.0
20273,29.0,2,50.0
2777,22.0,2,4.0
2059,14.0,2,2.0
3366,14.0,2,1.0
3367,14.0,1,1.0
11000,11.0,1,11.0
10974,11.0,1,1.0
10913,11.0,1,11.0
18820,11.0,1,1.0


## 10. Revisar la relación entre dormitorios y camas

Se calcula cuántos alojamientos tienen más dormitorios que camas. Esta condición puede señalar datos inconsistentes, pero no demuestra por sí sola que sean incorrectos: puede haber dormitorios sin camas registradas o información incompleta. Por eso es preferible inspeccionar y cuantificar los casos antes de eliminarlos.

La siguiente celda conserva la regla que se había probado originalmente. **Recomendación:** no ejecutar la eliminación automática hasta validar los casos y documentar el criterio; en esta versión la celda se deja como código comentado para que no descarte registros de manera irreversible durante una ejecución completa.


In [314]:
# Inspeccionar primero cuántos casos cumplen la condición
mask_bedrooms_gt_beds = df_dp["bedrooms"] > df_dp["beds"]
print("Filas con bedrooms > beds:", int(mask_bedrooms_gt_beds.sum()))
print("Porcentaje del dataset:", round(mask_bedrooms_gt_beds.mean() * 100, 2), "%")

# Después de revisar los casos, decidir si corresponde eliminarlos.
# No se recomienda borrar automáticamente solo por esta condición.
# df_dp = df_dp.loc[~mask_bedrooms_gt_beds].copy()


Filas con bedrooms > beds: 171
Porcentaje del dataset: 0.71 %


## 11. Verificar los casos restantes

Esta consulta permite comprobar qué filas todavía cumplen la condición `bedrooms > beds`. Si no se ha aplicado ningún filtro, debería mostrar los casos detectados; si se decide corregir o eliminar registros después de revisarlos, se puede volver a ejecutar para verificar el resultado.


In [315]:
df_dp.loc[df_dp["bedrooms"] > df_dp["beds"], ["bedrooms", "beds"]].sort_values(
    "bedrooms", ascending=False
)

,bedrooms,beds
18327,31.0,2.0
2777,22.0,4.0
2059,14.0,2.0
3367,14.0,1.0
3366,14.0,1.0
...,...,...
26712,2.0,1.0
27174,2.0,1.0
27274,2.0,1.0
28053,2.0,1.0


## 12. Renombrado de columnas para mejorar la legibilidad
En esta sección se renombran algunas columnas del DataFrame df_dp para utilizar nombres más cortos, claros y fáciles de manejar durante las siguientes etapas del análisis y el entrenamiento del modelo.

Se utiliza el método rename() de Pandas con el parámetro columns, mediante un diccionario que relaciona cada nombre original con su nuevo nombre.

Este cambio no modifica los valores de las columnas ni elimina información; únicamente modifica sus nombres.
- host_is_superhost → superhost: indica si el anfitrión es considerado superhost.
- price_quote_per_night → price: simplifica el nombre de la variable que representa el precio por noche.
- minimum_nights → min_nights: indica la cantidad mínima de noches requerida para reservar.
- maximum_nights → max_nights: indica la cantidad máxima de noches permitida para reservar.
El objetivo es facilitar la lectura del código y hacer que las variables sean más cómodas de utilizar en las siguientes etapas del proyecto.

In [316]:
# renombrar columnas para que sean más legibles
df_dp = df_dp.rename(
    columns={
        "host_is_superhost": "superhost",
        "price_quote_price_per_night": "price",
        "minimum_nights": "min_nights",
        "maximum_nights": "max_nights",
    }
)

## 13. Resumen y próximos pasos

Al finalizar este notebook se dispone de variables numéricas y binarias para describir el alojamiento, su ubicación, el tipo de propiedad, el estado de superhost, los baños y los servicios más comunes. También se documentaron posibles valores extremos para revisarlos según el contexto.

Antes de entrenar el modelo:

1. Comprobar los tipos de datos y los valores nulos de las columnas creadas.
2. Confirmar que el target sea `price` y excluir del conjunto de predictores cualquier columna que filtre directamente información del precio.
3. Separar entrenamiento y prueba antes de ajustar transformaciones que aprendan parámetros de los datos (por ejemplo, escalado, imputación o codificación final).
4. Guardar el dataset preprocesado y registrar las decisiones de limpieza para poder reproducir el flujo.


In [317]:
df_dp.to_csv("../data/processed/airbnb_dp.csv", index=False)